# Demostración del sistema de recomendación

Este notebook demuestra los nueve pasos del proyecto de Matrix Factorization.

## 1. Carga de los datos

Se cargan los dos CSV y se comprueban sus columnas esperadas.

In [ ]:
from src.data_loader import (
    build_mappings,
    compute_dataset_stats,
    load_data,
    temporal_train_test_split,
)
from src.model import MatrixFactorization
from src.evaluation import (
    compute_mae,
    compute_ndcg_at_k,
    compute_rmse,
)
from src.recommender import get_user_history, recommend_top_n

valoraciones, peliculas = load_data(
    "data/ratings.csv",
    "data/movies.csv",
)

print("Primeras filas de valoraciones:")
display(valoraciones.head())

print("Primeras filas de películas:")
display(peliculas.head())

## 2. Estadísticas del conjunto de datos

Se calcula el número de usuarios, películas, valoraciones y densidad.

In [ ]:
estadisticas = compute_dataset_stats(valoraciones)

for nombre, valor in estadisticas.items():
    print(f"{nombre}: {valor}")

## 3. División temporal 80/20

Las valoraciones más antiguas de cada usuario van a `train` y las más recientes a `test`.

In [ ]:
entrenamiento, prueba = temporal_train_test_split(
    valoraciones,
    train_ratio=0.8,
) 

cantidad_usuarios_train = entrenamiento["userId"].nunique()
cantidad_usuarios_prueba = prueba["userId"].nunique()
usuarios_sin_train = set(prueba["userId"]) - set(entrenamiento["userId"])

print("Valoraciones en train:", len(entrenamiento))
print("Valoraciones en test:", len(prueba))
print("Usuarios en train:", cantidad_usuarios_train)
print("Usuarios en test:", cantidad_usuarios_prueba)
print("Usuarios sin datos en train:", len(usuarios_sin_train))

print("Mínimo de valoraciones por usuario en train:")
print(entrenamiento.groupby("userId").size().min())

## 4. Construcción de mapeos y entrenamiento

Los mapeos se construyen exclusivamente desde `entrenamiento`.

In [ ]:
mapeos = build_mappings(entrenamiento)

modelo = MatrixFactorization(
    k=20,
    lr=0.005,
    lambda_=0.02,
    epochs=20,
    seed=42,
)

historial_mse = modelo.fit(entrenamiento, mapeos)

print("Usuarios internos:", len(mapeos["user2idx"]))
print("Películas internas:", len(mapeos["movie2idx"]))

## 5. Curva de aprendizaje MSE

Cada punto representa el MSE calculado al finalizar una época.

In [ ]:
import matplotlib.pyplot as plt

epocas = list(range(1, len(historial_mse) + 1))

for epoca, mse in zip(epocas, historial_mse):
    print(f"Época {epoca:2d}: MSE = {mse:.6f}")

plt.figure(figsize=(8, 4))
plt.plot(epocas, historial_mse, marker="o")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.title("Curva de aprendizaje del modelo")
plt.grid(True)
plt.show()

## 6. Evaluación sobre el conjunto test

Se calculan RMSE, MAE y NDCG@10.

In [ ]:
import numpy as np

valores_reales = prueba["rating"].to_numpy(dtype=float)
pares_usuario_pelicula = list(
    zip(prueba["userId"].tolist(), prueba["movieId"].tolist())
)
valores_predichos = np.array([
    modelo.predict_one(usuario, pelicula)
    for usuario, pelicula in pares_usuario_pelicula
])

rmse = compute_rmse(valores_reales, valores_predichos)
mae = compute_mae(valores_reales, valores_predichos)
ndcg = compute_ndcg_at_k(prueba, modelo, k=10)

print("RMSE:", rmse)
print("MAE:", mae)
print("NDCG@10:", ndcg)
print("Cumple RMSE < 0.95:", rmse < 0.95)
print("MAE aproximadamente 0.67:", abs(mae - 0.67) < 0.03)

## 7. Comparación de valoraciones reales y predichas

Se seleccionan cinco parejas usuario-película del conjunto test.

In [ ]:
parejas_comparacion = prueba.iloc[:5]

tabla_comparacion = parejas_comparacion[
    ["userId", "movieId", "rating"]
].copy()
tabla_comparacion["predicted_rating"] = [
    modelo.predict_one(fila["userId"], fila["movieId"])
    for _, fila in parejas_comparacion.iterrows()
]

print("Predicciones dentro de [0.5, 5.0]:",
      tabla_comparacion["predicted_rating"].between(0.5, 5.0).all())
display(tabla_comparacion)

## 8. Recomendaciones Top-10 para tres usuarios

Se eligen tres usuarios conocidos y se excluyen las películas vistas en `train`.

In [ ]:
usuarios_conocidos = entrenamiento["userId"].drop_duplicates().head(3).tolist()

for usuario in usuarios_conocidos:
    print("Usuario:", usuario)
    recomendaciones = recommend_top_n(
        modelo,
        peliculas,
        user_id=usuario,
        n=10,
    )
    display(recomendaciones)

## 9. Historial de valoraciones de un usuario

Se muestra el historial unido con título y género, ordenado por `rating`.

In [ ]:
usuario_historial = usuarios_conocidos[0]
historial_usuario = get_user_history(
    valoraciones,
    peliculas,
    user_id=usuario_historial,
)

print("Usuario:", usuario_historial)
display(historial_usuario)